# FraSoHome Agents Foundry - Notebook de demo interactiva

Este notebook permite ejecutar paso a paso los módulos Python del repositorio durante la sesión práctica.

La ruta de demo recomendada es:

1. Validar assets del caso, KB y CSVs.
2. Mostrar la base de conocimiento que se cargará en File Search.
3. Perfilar los datos localmente con pandas.
4. Previsualizar la creación de agentes en Foundry.
5. Crear agentes en Foundry, si el entorno está configurado.
6. Ejecutar las tres demos: Knowledge, Data Quality y Orquestador multiagente.

Las celdas que llaman a Azure están protegidas por `RUN_LIVE_FOUNDRY = False` para evitar ejecuciones accidentales.

## 0. Preparación

Antes de abrir el notebook, se recomienda crear el entorno:

```powershell
python -m venv .venv
.\.venv\Scripts\Activate.ps1
pip install -r requirements.txt
pip install -e .
copy .env.example .env
az login
```

Configura `.env` con:

```text
PROJECT_ENDPOINT=https://<resource>.services.ai.azure.com/api/projects/<project>
MODEL_DEPLOYMENT=<deployment-name>
```

In [1]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "src" / "frasohome_agents").exists():
    # Useful if the notebook is opened from notebooks/ instead of repo root.
    REPO_ROOT = Path.cwd().parent

sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)

print("Repo root:", REPO_ROOT)
print("Python:", sys.version)

Repo root: c:\Repos\frasohome-agents-foundry
Python: 3.12.6 (tags/v3.12.6:a4a2d2b, Sep  6 2024, 20:11:23) [MSC v.1940 64 bit (AMD64)]


In [2]:
from dotenv import load_dotenv

load_dotenv(REPO_ROOT / ".env")

# Safety switch: keep False during rehearsals unless you want to call Microsoft Foundry.
RUN_LIVE_FOUNDRY = True

print("RUN_LIVE_FOUNDRY =", RUN_LIVE_FOUNDRY)
print("PROJECT_ENDPOINT set:", bool(os.getenv("PROJECT_ENDPOINT")))
print("MODEL_DEPLOYMENT:", os.getenv("MODEL_DEPLOYMENT", "<not set>"))

RUN_LIVE_FOUNDRY = True
PROJECT_ENDPOINT set: True
MODEL_DEPLOYMENT: gpt-5-mini


## 1. Cargar configuración y validar assets

Esta sección no llama a Azure. Comprueba que el caso, la KB y los CSV están donde el código espera.

In [3]:
from frasohome_agents.settings import Settings, ensure_output_dir
from frasohome_agents.assets import assert_assets, knowledge_files, kb_markdown_files, csv_files

settings = Settings.load()
ensure_output_dir(settings)
assert_assets(settings)

print("Case dir:", settings.case_dir)
print("Output dir:", settings.output_dir)
print("Knowledge files:", len(knowledge_files(settings)))
print("KB policy files:", len(kb_markdown_files(settings)))
print("CSV files:", len(csv_files(settings)))

Case dir: C:\Repos\frasohome-agents-foundry\case
Output dir: C:\Repos\frasohome-agents-foundry\outputs
Knowledge files: 10
KB policy files: 7
CSV files: 11


In [5]:
print("Documentos que se cargarán en File Search:")
for path in knowledge_files(settings):
    print("-", path.relative_to(REPO_ROOT))

print("CSVs que se cargarán en Code Interpreter:")
for path in csv_files(settings):
    print("-", path.relative_to(REPO_ROOT))

Documentos que se cargarán en File Search:
- case\fraso_home_caso.md
- case\fraso_home_storytelling_foundry.md
- case\kb\README.md
- case\kb\FS-KB-01_Politica_Devoluciones_v1.3_Vigente.md
- case\kb\FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.md
- case\kb\FS-KB-04_Manual_Tienda_Caja_y_Pagos_Mixtos_v2.1.md
- case\kb\FS-KB-05_Guia_Conciliacion_Pagos_Ecommerce_v1.4.md
- case\kb\FS-KB-06_Taxonomia_Catalogo_y_Reglas_SKU_v1.2.md
- case\kb\FS-KB-07_Guia_Fidelizacion_CRM_v1.1.md
- case\kb\FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.md
CSVs que se cargarán en Code Interpreter:
- case\data\crm.csv
- case\data\devoluciones_online.csv
- case\data\devoluciones_tienda.csv
- case\data\fact_transacciones.csv
- case\data\lineas_pedido.csv
- case\data\pagos_tienda.csv
- case\data\pedidos.csv
- case\data\productos.csv
- case\data\stock_diario.csv
- case\data\tiendas.csv
- case\data\ventas_pos.csv


## 2. Mostrar el caso y la KB

Estas celdas ayudan a explicar el storytelling y qué políticas gobiernan las respuestas normativas del agente Knowledge.

In [6]:
from IPython.display import Markdown, display

case_md = (REPO_ROOT / "case" / "fraso_home_storytelling_foundry.md").read_text(encoding="utf-8")
display(Markdown(case_md[:4000]))

# Caso y storytelling FraSoHome para Microsoft Foundry

Fuente: extracción narrativa de `Global AI Chapter - Agentes_Microsoft_Foundry_FraSoHome.pptx`.

## Hilo conductor

FraSoHome es un retailer omnicanal de hogar y decoración con datos repartidos entre CRM, POS, e-commerce, ERP, devoluciones y stock. La historia de la demo no intenta demostrar que la IA “lo hace todo”, sino que un sistema agentic bien diseñado reduce fricción, mejora trazabilidad y convierte datos desordenados en decisiones accionables.

## Problema de negocio

FraSoHome necesita una vista única y fiable para responder cuatro preguntas sencillas pero difíciles de contestar con datos fragmentados:

- ¿Qué vendemos?
- ¿A quién vendemos?
- ¿En qué canal vendemos?
- ¿Con qué margen y con qué patrón de devolución?

Los dolores visibles en la presentación son nulos, duplicados, fechas y claves inconsistentes, productos sin mapeo, devoluciones cruzadas entre canales, stock negativo o fuera de rango y decisiones poco fiables por falta de integración.

## Tesis de la sesión

El valor de Microsoft Foundry aparece cuando conectamos conocimiento, datos y acción sobre un proceso concreto. El agente no sustituye las reglas de negocio ni las políticas: las hace accesibles, trazables y operativas. La pregunta de diseño es qué debe razonar el agente y qué debe quedar gobernado por reglas, validaciones y herramientas deterministas.

## Arquitectura narrativa

La presentación propone un sistema multiagente coordinado por un orquestador:

- **Orquestador:** enruta la intención, mantiene estado y sintetiza la respuesta final.
- **Knowledge:** responde con grounding sobre documentos, políticas, catálogo y promociones.
- **Data Quality:** perfila CSVs, detecta anomalías y propone limpieza.
- **Customer:** calcula métricas de cliente, RFM, preferencias y experiencia.
- **Operations:** analiza stock, pedidos, tiendas y posibles impactos operativos.
- **Returns:** explica devoluciones, motivos, tasas y patrones por canal/categoría.
- **Storyteller:** convierte evidencias técnicas en una recomendación ejecutiva clara.

El contrato entre agentes se basa en salidas estructuradas:

```json
{
  "hallazgos": [],
  "evidencias": [],
  "riesgos": [],
  "siguiente_accion": "..."
}
```

## Recorrido de demostración

### Escena 1: Preguntamos a FraSoHome

Demo de agente de conocimiento con File Search. El usuario pregunta: “Un cliente compró un sofá online, quiere devolverlo en tienda y usó un cupón. ¿Qué pasos debe seguir atención al cliente?”.

La salida esperada es una respuesta breve con pasos operativos, evidencia del documento y aviso explícito si falta un dato crítico.

### Escena 2: Miramos los datos

Demo de Code Interpreter sobre los CSVs de FraSoHome. El agente lee CRM, pedidos, líneas, devoluciones, POS, stock y productos; perfila datos; detecta anomalías; sugiere limpieza; y resume impacto en un Data Quality Report.

El cierre narrativo es: “Ahora ya tenemos una lista priorizada de problemas antes de construir features o dashboards”.

### Escena 3: Orquestamos especialistas

Demo multiagente con una pregunta ambigua: “¿Por qué están subiendo las devoluciones online en iluminación y qué haríamos esta semana?”.

El router detecta que la intención mezcla calidad de datos, operaciones, cliente y devoluciones. Cada especialista aporta evidencias y Storyteller devuelve una decisión con cinco piezas: causa probable, evidencia, impacto, acción de 7 días y métrica de seguimiento.

## Mensaje final para la audiencia

Foundry aporta un lugar común para pasar de idea a producción: agentes, herramientas, workflows, seguridad, observabilidad, evaluación y despliegue dentro de Azure. La ruta recomendada es empezar con prompt agents y workflows para que el proceso sea visible, y mover a SDK/Agent Framework cuando hagan falta lógica propia, estado, tests, CI/CD o integración avanzada.


In [20]:
kb_index = (REPO_ROOT / "case" / "kb" / "README.md").read_text(encoding="utf-8")
display(Markdown(kb_index))

# Base de conocimiento FraSoHome

Políticas y guías convertidas a Markdown desde los DOCX de `case/kb`.

| Documento | Markdown | Contenido | Tablas |
|---|---|---|---|
| `FS-KB-01_Politica_Devoluciones_v1.3_Vigente.docx` | `FS-KB-01_Politica_Devoluciones_v1.3_Vigente.md` | Política de devoluciones (omnichannel) - FraSoHome | 6 |
| `FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.docx` | `FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.md` | Diccionario de KPI y reglas de cálculo - FraSoHome | 2 |
| `FS-KB-04_Manual_Tienda_Caja_y_Pagos_Mixtos_v2.1.docx` | `FS-KB-04_Manual_Tienda_Caja_y_Pagos_Mixtos_v2.1.md` | Manual de tienda: cierre de caja y pagos mixtos - FraSoHome | 2 |
| `FS-KB-05_Guia_Conciliacion_Pagos_Ecommerce_v1.4.docx` | `FS-KB-05_Guia_Conciliacion_Pagos_Ecommerce_v1.4.md` | Guía de conciliación de pagos (E-commerce) - FraSoHome | 3 |
| `FS-KB-06_Taxonomia_Catalogo_y_Reglas_SKU_v1.2.docx` | `FS-KB-06_Taxonomia_Catalogo_y_Reglas_SKU_v1.2.md` | Taxonomía de catálogo y reglas de SKU - FraSoHome | 3 |
| `FS-KB-07_Guia_Fidelizacion_CRM_v1.1.docx` | `FS-KB-07_Guia_Fidelizacion_CRM_v1.1.md` | Guía de fidelización CRM (FraSoHome Rewards) | 3 |
| `FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.docx` | `FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.md` | FAQ interna FraSoHome - Operaciones y Atención | 1 |

Uso recomendado en Foundry:

- Cargar todos los Markdown de `case/kb` en el vector store de `frasohome-knowledge`.
- Usar `case/kb/README.md` como índice documental.
- Mantener los DOCX originales como fuente de archivo y los Markdown como versión optimizada para File Search.



## 3. Perfilado local de datos

Este perfilado usa pandas y no requiere Foundry. Sirve para enseñar el diagnóstico antes de ejecutar Code Interpreter.

In [4]:
from frasohome_agents.local_data_quality import profile_all

local_report = profile_all(settings)
print("Archivos perfilados:", len(local_report["resumen"]))
print("Reporte JSON:", settings.output_dir / "local_data_quality_report.json")
print("Reporte Markdown:", settings.output_dir / "local_data_quality_report.md")

Archivos perfilados: 11
Reporte JSON: C:\Repos\frasohome-agents-foundry\outputs\local_data_quality_report.json
Reporte Markdown: C:\Repos\frasohome-agents-foundry\outputs\local_data_quality_report.md


In [5]:
import pandas as pd

summary_df = pd.DataFrame(local_report["resumen"])[["archivo", "filas", "columnas", "nulos_totales", "duplicados"]]
summary_df.sort_values(["duplicados", "nulos_totales"], ascending=False)

,archivo,filas,columnas,nulos_totales,duplicados
10,ventas_pos.csv,2521,19,4534,30
8,stock_diario.csv,1940,9,2015,14
5,pagos_tienda.csv,1646,12,2010,10
4,lineas_pedido.csv,1949,12,1875,8
6,pedidos.csv,656,16,831,2
0,crm.csv,103,20,214,1
3,fact_transacciones.csv,4973,59,47653,0
2,devoluciones_tienda.csv,211,16,516,0
1,devoluciones_online.csv,224,12,243,0
7,productos.csv,101,18,18,0


In [6]:
from IPython.display import Markdown, display

report_md = (settings.output_dir / "local_data_quality_report.md").read_text(encoding="utf-8")
display(Markdown(report_md))

# Data Quality Report local - FraSoHome

Este informe se calcula localmente con pandas para validar la demo antes de ejecutar Code Interpreter en Foundry.

| Archivo | Filas | Columnas | Nulos totales | Duplicados |
|---|---:|---:|---:|---:|
| crm.csv | 103 | 20 | 214 | 1 |
| devoluciones_online.csv | 224 | 12 | 243 | 0 |
| devoluciones_tienda.csv | 211 | 16 | 516 | 0 |
| fact_transacciones.csv | 4973 | 59 | 47653 | 0 |
| lineas_pedido.csv | 1949 | 12 | 1875 | 8 |
| pagos_tienda.csv | 1646 | 12 | 2010 | 10 |
| pedidos.csv | 656 | 16 | 831 | 2 |
| productos.csv | 101 | 18 | 18 | 0 |
| stock_diario.csv | 1940 | 9 | 2015 | 14 |
| tiendas.csv | 8 | 19 | 6 | 0 |
| ventas_pos.csv | 2521 | 19 | 4534 | 30 |

## Acciones priorizadas

- Resolver claves de producto y cliente sin correspondencia antes de integrar la fact table.
- Eliminar o consolidar duplicados en ventas POS, stock diario, líneas de pedido, pedidos y CRM.
- Estandarizar fechas y revisar registros no parseables o fuera del periodo de operación.
- Normalizar importes, cantidades, descuentos y stocks antes de calcular KPIs.
- Aplicar reglas de la KB para ventas netas, devoluciones, SKU, pagos y fidelización.


## 4. Prompts e instrucciones de agentes

Usa estas celdas para enseñar el diseño de instrucciones antes de crear los agentes.

In [7]:
from frasohome_agents import prompts

print("Knowledge prompt:", prompts.KNOWLEDGE_PROMPT)
print("Data Quality prompt:", prompts.DATA_QUALITY_PROMPT)
print("Multi-agent prompt:", prompts.MULTI_AGENT_PROMPT)

Knowledge prompt: Un cliente compró un sofá online, quiere devolverlo en tienda y usó un cupón. ¿Qué pasos debe seguir atención al cliente?
Data Quality prompt: Analiza los CSV de FraSoHome. Genera un Data Quality Report con: filas y columnas por archivo, nulos críticos, duplicados, claves sin correspondencia, fechas fuera de rango, importes/cantidades anómalas y cinco acciones de limpieza priorizadas. Usa el diccionario de KPI, la taxonomía SKU y la política de devoluciones de la KB como referencia de reglas. Devuelve tabla resumen y recomendaciones.

Multi-agent prompt: ¿Por qué están subiendo las devoluciones online en iluminación y qué haríamos esta semana?


In [8]:
print(prompts.KNOWLEDGE_INSTRUCTIONS[:1600])

Eres el agente Knowledge de FraSoHome.

Tu función es responder preguntas operativas sobre FraSoHome usando solo los documentos conectados al agente.

Prioridad documental:
1. Políticas y guías vigentes de la KB.
2. FAQ interna de operaciones y atención.
3. Documento narrativo del caso.
4. Storytelling de la presentación.

Reglas:
- No inventes políticas, excepciones ni métricas.
- Si falta una política formal o hay conflicto entre documentos, dilo de forma explícita.
- Responde con pasos operativos breves.
- Incluye evidencia documental o la parte del caso en la que te basas.
- Incluye incertidumbres y siguiente acción.

Formato de respuesta:
1. Respuesta
2. Evidencia
3. Incertidumbres
4. Siguiente acción



## 5. Previsualizar creación de agentes

`dry_run=True` no llama a Azure. Muestra qué agentes, herramientas y archivos se usarán.

In [9]:
from frasohome_agents.create_agents import create_all_agents

agent_plan = create_all_agents(dry_run=True)
agent_plan.keys()

{
  "frasohome-knowledge": {
    "tool": "FileSearchTool",
    "files": [
      "C:\\Repos\\frasohome-agents-foundry\\case\\fraso_home_caso.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\fraso_home_storytelling_foundry.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\kb\\README.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\kb\\FS-KB-01_Politica_Devoluciones_v1.3_Vigente.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\kb\\FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\kb\\FS-KB-04_Manual_Tienda_Caja_y_Pagos_Mixtos_v2.1.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\kb\\FS-KB-05_Guia_Conciliacion_Pagos_Ecommerce_v1.4.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\kb\\FS-KB-06_Taxonomia_Catalogo_y_Reglas_SKU_v1.2.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\kb\\FS-KB-07_Guia_Fidelizacion_CRM_v1.1.md",
      "C:\\Repos\\frasohome-agents-foundry\\case\\kb\\FS-KB-09_FAQ_Operaciones_y_Atencion_v1.0.md"
    ]
  },
  "frasohome-data-quality": {
    "tool": "CodeInterpreterTool",
    "files": [
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\crm.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\devoluciones_online.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\devoluciones_tienda.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\fact_transacciones.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\lineas_pedido.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\pagos_tienda.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\pedidos.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\productos.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\stock_diario.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\tiendas.csv",
      "C:\\Repos\\frasohome-agents-foundry\\case\\data\\ventas_pos.csv"
    ]
  },
  "frasohome-returns": {
    "tool": "FileSearchTool optional",
    "instructions": "Eres el agente Returns de FraSoHome.\n\nAnaliza preguntas sobre devoluciones online y tienda. Usa la política de devoluciones vigente, la FAQ interna y, cuando aplique, el manual de tienda para pagos mixtos. Cuando haya datos disponibles a través de otro agente o del workflow, interpreta motivos, canales, categorías, tasas e impacto.\n\nDevuelve siempre JSON:\n{\n  \"agent\": \"returns\",\n  \"hallazgos\": [],\n  \"evidencias\": [],\n  \"riesgos\": [],\n  \"confianza\": 0.0\n}\n"
  },
  "frasohome-operations": {
    "tool": "FileSearchTool optional",
    "instructions": "Eres el agente Operations de FraSoHome.\n\nAnaliza posibles causas operativas relacionadas con stock, tiendas, pedidos, logística, disponibilidad, pagos, conciliación ecommerce, tienda, taxonomía SKU y canal. Usa el manual de tienda, la guía de conciliación, la taxonomía de catálogo y el diccionario KPI como referencia. Señala hipótesis operativas y qué dato hace falta para validarlas.\n\nDevuelve siempre JSON:\n{\n  \"agent\": \"operations\",\n  \"hallazgos\": [],\n  \"evidencias\": [],\n  \"riesgos\": [],\n  \"confianza\": 0.0\n}\n"
  },
  "frasohome-storyteller": {
    "tool": "none",
    "instructions": "Eres el agente Storyteller de FraSoHome.\n\nTu función es convertir evidencias de agentes especialistas en una recomendación ejecutiva clara. No añadas cifras nuevas. No ocultes incertidumbres.\n\nDevuelve JSON válido:\n{\n  \"pregunta\": \"\",\n  \"causa_probable\": \"\",\n  \"evidencias\": [\n    {\n      \"fuente\": \"\",\n      \"calculo\": \"\",\n      \"valor\": \"\"\n    }\n  ],\n  \"riesgos\": [],\n  \"accion_7_dias\": \"\",\n  \"metrica_seguimiento\": \"\",\n  \"requiere_validacion_humana\": true\n}\n"
  }
}

dict_keys(['frasohome-knowledge', 'frasohome-data-quality', 'frasohome-returns', 'frasohome-operations', 'frasohome-storyteller'])

## 6. Crear agentes en Microsoft Foundry

Activa `RUN_LIVE_FOUNDRY = True` solo cuando:

- `.env` contiene `PROJECT_ENDPOINT` y `MODEL_DEPLOYMENT`.
- Has ejecutado `az login`.
- Tu usuario tiene permisos para crear agentes y cargar archivos.

Esta celda intenta crear:

- `frasohome-knowledge` con File Search.
- `frasohome-data-quality` con Code Interpreter.
- `frasohome-returns` con File Search.
- `frasohome-operations` con File Search.
- `frasohome-storyteller` sin herramientas.

In [10]:
if RUN_LIVE_FOUNDRY:
    created = create_all_agents(dry_run=False)
    print(created.keys())
else:
    print("Saltado. Cambia RUN_LIVE_FOUNDRY = True para crear agentes en Foundry.")

Uploaded KB file: fraso_home_caso.md

Uploaded KB file: fraso_home_storytelling_foundry.md

Uploaded KB file: README.md

Uploaded KB file: FS-KB-01_Politica_Devoluciones_v1.3_Vigente.md

Uploaded KB file: FS-KB-03_Diccionario_KPI_Reglas_Calculo_v1.0.md

...

Uploaded CSV: stock_diario.csv

Uploaded CSV: tiendas.csv

Uploaded CSV: ventas_pos.csv

Wrote C:\Repos\frasohome-agents-foundry\outputs\foundry_created_agents.json

dict_keys(['frasohome-knowledge', 'frasohome-data-quality', 'frasohome-returns', 'frasohome-operations', 'frasohome-storyteller', 'vector_store_id'])


## 7. Demo 1 - Knowledge

Pregunta canónica:

> Un cliente compró un sofá online, quiere devolverlo en tienda y usó un cupón. ¿Qué pasos debe seguir atención al cliente?

In [11]:
from frasohome_agents.run_agents import run_knowledge

if RUN_LIVE_FOUNDRY:
    knowledge_answer = run_knowledge()
else:
    print("Saltado. Esta celda llama al agente Knowledge en Foundry.")

Tool calls

- file_search_call: completed

1) Respuesta — pasos operativos para Atención al Cliente (breve)
- 1. Solicitar y comprobar: número de pedido + confirmación de entrega y método(s) de pago. Verificar que la 
devolución se solicita dentro del plazo online (45 días desde la entrega). 
- 2. Identificar producto voluminoso / estado montaje: considerar el sofá como “voluminoso” y, si está montado, 
pedir validación de Operaciones antes de aceptar la devolución voluntaria.  
- 3. Localizar compra en tienda: pedir ticket o localizar por fidelización (email/teléfono). Si no hay 
ticket/fidelización, no procesar sin aprobación del Store Manager. 
- 4. Inspección en tienda: revisar estado, anotar motivo (códigos R01–R06) y condición. Registrar en el sistema el 
motivo y la condición. 
- 5. Reembolso: devolver al método de pago original; en pagos mixtos, reembolsar por tramos siguiendo la regla 
(primero el tramo correspondiente). Ejecutar reembolso en POS y entregar justificante. Tiempo objetivo: en tienda, 
inmediato cuando sea posible o hasta 5 días si requiere validación.  
- 6. Costes y gastos de envío: si es devolución voluntaria de compra online, los gastos de envío iniciales no se 
reembolsan; los costes de recogida de voluminosos pueden repercutirse según tarifa. Informar al cliente. 
- 7. Si hay sospecha de incidencia (defecto/daño) o fraude, seguir el flujo de incidencias/escalado (Prevención de 
pérdidas / Operaciones). 

2) Evidencia (documental / extractos usados)
- Plazo devoluciones online: 45 días naturales desde la entrega (FS‑KB‑01).   
- Sofás = producto voluminoso; montado / voluminoso requiere validación de Operaciones (FS‑KB‑01; FAQ).    
- Flujo en tienda: localizar compra, validar política/estado, registrar motivo, ejecutar reembolso al método 
original; reglas para pagos mixtos descritas en el manual de tienda (FS‑KB‑04).    
- Gastos: gastos de envío no reembolsables en devoluciones voluntarias; costes de recogida de voluminosos pueden 
repercutirse (FS‑KB‑01).   
- Notación operacional: el POS registra métodos como tarjeta, efectivo, cupón; el caso de uso menciona cupón como 
método posible (dataset/operaciones). (Caso operativo). 

3) Incertidumbres / huecos en la documentación
- No hay una regla explícita en los documentos consultados sobre el tratamiento exacto cuando el pago original fue 
un cupón (p. ej. reactivar/reemitir el mismo cupón, emitir nuevo código, o dar saldo en cuenta). Los documentos 
definen la regla general “reembolso al método de pago original” pero no detallan procedimientos específicos para 
cupones.  
- Necesitamos saber: (a) si el sofá está montado; (b) si el pago fue 100% con cupón o cupón + otro método; (c) si 
hay ticket/fidelización disponible. Sin esos datos no se puede decidir definitivamente la acción sobre el cupón ni 
confirmar tiempos exactos de reembolso.

4) Siguiente acción (qué debe hacer Atención al Cliente ahora)
- 1) Pedir al cliente: número de pedido, comprobante de entrega, indicar si el sofá está montado y mostrar cómo 
pagó (captura/confirmación si tiene).  
- 2) Si es voluntaria y dentro de plazo: pedir que lo traiga a tienda para inspección; avisar que voluminosos 
requieren validación de Operaciones (abrir incidencia a Operaciones si está montado). 
- 3) Si el pago incluyó un cupón: abrir incidencia a Finanzas/Operaciones para confirmar política interna de 
reemisión o devolución de cupón (contacto Finanzas / Store Manager si procede). Sugerencia operacional inmediata: 
intentar identificar en el pedido el método original y, si el POS/ERP no permite reembolsar a cupón 
automáticamente, escalar a finanzas (finanzas@frasohome) o al manager de tienda.  
- 4) Registrar todo en el sistema (motivo R0x, evidencia fotos si hay daño, referencia a la orden) y confirmar al 
cliente los plazos esperados (reembolso en tienda: generalmente inmediato o hasta 5 días; si se necesita 
intervención de almacén/operaciones/finanzas, informar del plazo estimado). 
- 5) Si detectas patrón sospechoso o falta de ticke

Wrote C:\Repos\frasohome-agents-foundry\outputs\knowledge_response.md

## 8. Demo 2 - Data Quality con Code Interpreter

Esta celda llama al agente `frasohome-data-quality`. El agente debe usar Code Interpreter para calcular métricas desde CSVs.

In [12]:
from frasohome_agents.run_agents import run_data_quality

if RUN_LIVE_FOUNDRY:
    dq_answer = run_data_quality()
else:
    print("Saltado. Esta celda llama a Code Interpreter en Foundry.")

Tool calls

- code_interpreter_call: completed

- code_interpreter_call: completed

- code_interpreter_call: completed

- code_interpreter_call: completed

...

- code_interpreter_call: completed

- code_interpreter_call: completed

- code_interpreter_call: completed

Data Quality Report — FraSoHome
Resumen ejecutable y recomendaciones (análisis realizado con Python sobre los CSV entregados)

1) Tabla resumen por archivo
Formato: archivo — filas, columnas, nulos totales, filas duplicadas (full-row), PK candidato, filas con PK 
duplicada (si aplica)

- pagos_tienda.csv — 1646 filas, 12 cols, 2010 nulos, 20 filas duplicadas, PK candidato: payment_id, filas con PK 
duplicada: 30
- lineas_pedido.csv — 1949 filas, 12 cols, 1875 nulos, 16 filas duplicadas, PK candidato: order_line_id, filas con 
PK duplicada: 39
- devoluciones_online.csv — 224 filas, 12 cols, 243 nulos, 0 filas duplicadas, PK candidato: order_line_id, filas 
con PK duplicada: 13
- fact_transacciones.csv — 4973 filas, 59 cols, 47653 nulos, 0 filas duplicadas, PK candidato: return_id, filas con
PK duplicada: 4574
- crm.csv — 103 filas, 20 cols, 214 nulos, 2 filas duplicadas, PK candidato: customer_id, filas con PK duplicada: 4
- devoluciones_tienda.csv — 211 filas, 16 cols, 516 nulos, 0 filas duplicadas, PK candidato: return_id, filas con 
PK duplicada: 10
- ventas_pos.csv — 2521 filas, 19 cols, 4534 nulos, 60 filas duplicadas, PK candidato: customer_id (nota: el PK 
real debería ser ticket_line_id), filas con PK duplicada: 2487
- pedidos.csv — 656 filas, 16 cols, 831 nulos, 4 filas duplicadas, PK candidato: customer_id (nota: el PK real 
debería ser order_id), filas con PK duplicada: 621
- tiendas.csv — 8 filas, 19 cols, 6 nulos, 0 filas duplicadas, PK candidato: store_id, filas con PK duplicada: 0
- stock_diario.csv — 1940 filas, 9 cols, 2015 nulos, 28 filas duplicadas, PK candidato: product_id, filas con PK 
duplicada: 1899
- productos.csv — 101 filas, 18 cols, 18 nulos, 0 filas duplicadas, PK candidato: product_id, filas con PK 
duplicada: 2

(Comentarios: algunos archivos contienen muchas columnas no parseadas en tipos correctos, por eso cuentan altos 
nulos totales en columnas derivadas; en fact_transacciones hay numerosas columnas convertidas/derivadas con nulos 
por fuente y parsing.)

2) Campos críticos con nulos (resumen — sólo campos clave con >0 nulos)
- pagos_tienda: fecha_pago — 123 parse-fail / nulos; importe_pagado (como texto) tiene parse-errors al convertir a 
numérico (ver sección numeric anomalies).
- lineas_pedido: precio_unitario, descuento_importe, importe_linea contienen valores no numéricos/parseables; 
order_id/order_line_id con algunos nulos (ver tabla crítica).
- devoluciones_online: importe_reembolsado y campos de fecha parseables pero hay 13 devoluciones que no tienen 
order_line_id coincidente con lineas_pedido.
- fact_transacciones: customer_id_std — 1749 nulos (no normalizado / no mapeado a CRM); product_id_std — 39 nulos; 
fecha_movimiento — 19 nulos; precio_unitario_num — 253 nulos.
- crm: fecha_alta_programa — 13 nulos; email — 7 nulos; puntos_acumulados y tier_fidelizacion con algunos nulos.
- devoluciones_tienda: fecha_devolucion — 41 parse-errors/nulos; importe_reembolsado parseado parcialmente.
- ventas_pos: fecha_hora — 170 parse-errors/nulos; precio_unitario/descuento_importe/importe_linea con formatos no 
numéricos en algunos registros.
- pedidos: fecha_pedido — parseada correctamente en la mayoría, pero importe_total y gastos_envio en algunos 
registros con formatos inconsistentes.
- tiendas: fecha_apertura — 1 parse-error; codigo_postal en algunos con NaN (ej. tienda Bilbao) y lat/lon con 
valores inválidos (latitud 91.0).
- stock_diario: fecha — 54 parse-errors; stock_cierre en formato no numérico en varias filas.
- productos: fecha_alta_catalogo — 1 parse-error; precio_venta/coste_unitario en texto en algunos registros (1 
precio_venta == 0, 1 coste_unitario == 0).

3) Duplicados y limpiezas de clave primaria
- Productos: 2 filas con mismo product_id (probable duplicado de catálogo). Revisar y deduplicar por product_id 
(conservar fila más completa).
- Tiendas: hay repeticiones de la misma tienda (S003 / "S003 ") con diferencias de casing/spaces y otros metadatos 
— hay duplicados semá

Wrote C:\Repos\frasohome-agents-foundry\outputs\data_quality_report.md

## 9. Demo 3 - Orquestador multiagente en código

Esta celda reproduce el workflow visual desde Python:

1. Knowledge aporta contexto normativo.
2. Data Quality aporta evidencia de datos.
3. Returns interpreta devoluciones.
4. Operations interpreta causas operativas.
5. Storyteller sintetiza la acción de 7 días.

In [13]:
from frasohome_agents.orchestrator import run_orchestrator

if RUN_LIVE_FOUNDRY:
    orchestrator_result = run_orchestrator()
else:
    print("Saltado. Esta celda invoca varios agentes en Foundry.")

{
  "pregunta": "¿Por qué están subiendo las devoluciones online en iluminación y qué haríamos esta semana?",
  "causa_probable": "Causa mixta: (A) problemas de producto/calidad y faltantes (defectuoso, faltan piezas), (B) embalaje/transporte que causa daños en tránsito, (C) devoluciones por ajuste de expectativa/medidas (no encaja) y (D) sesgo/inflación de la métrica por problemas de calidad de datos y mapeo omnicanal (product_id faltantes, order_line_id nulos, formatos de importe inconsistentes).",
  "evidencias": [
    {
      "fuente": "devoluciones_online (análisis Data Quality / returns)",
      "calculo": "Porcentaje devoluciones Iluminación = devoluciones_Iluminación / devoluciones_online_total",
      "valor": "53 / 224 = 23.7% (devoluciones Iluminación sobre total devoluciones online)"
    },
    {
      "fuente": "devoluciones_online (top SKUs)",
      "calculo": "Conteo devoluciones por SKU (filtro categoría Iluminación)",
      "valor": "Top SKUs: P1038(5), P1086(4), P1057(3), P1016(3), P1013(3), P1094(3), P1008(3), P1037(3), P1015(2), P1046(2)"
    },
    {
      "fuente": "devoluciones_online / returns (motivos registrados)",
      "calculo": "Recuento motivos en devoluciones categoría Iluminación",
      "valor": "Defectuoso 12, Faltan piezas 8, No encaja/medidas 8, Daño en transporte 4, NaN 5, Otros variado"
    },
    {
      "fuente": "fact_transacciones (consolidada)",
      "calculo": "Serie mensual devoluciones ONLINE en Iluminación (ejemplos)",
      "valor": "Ej.: 2025-03 ONLINE 0, 2025-05 ONLINE 6, 2025-06 ONLINE 4, 2025-11 ONLINE 5 — picos reproducibles en fact_table"
    },
    {
      "fuente": "data_quality report",
      "calculo": "Errores y nulos detectados en tablas fuente",
      "valor": "32 devoluciones refieren product_id no en catálogo; 9–12 devoluciones con order_line_id nulo o sin match; 5 fechas fuera de rango (ej. 2031-07-01); formato mixto de importes y reembolsos negativos detectados"
    },
    {
      "fuente": "Política de devoluciones / Guía conciliación (FS-KB-01, FS-KB-05)",
      "calculo": "Reglas y SLAs aplicables",
      "valor": "Plazo solicitud devolución online 45 días; objetivo ejecución reembolso e‑commerce 7–10 días; conciliación D+1 y discrepancias >500€ investigadas same‑day"
    }
  ],
  "riesgos": [
    "Tomar acciones estratégicas (recalls, promociones, cambios de proveedor) sin limpiar datos puede llevar a decisiones erróneas por métricas infladas o mal atribuidas.",
    "Si la causa real es embalaje/transportista, los costes logísticos (recogida, reposición, reenvío) continuarán hasta corregir embalaje o carrier.",
    "Errores en conciliación o importes mal parseados (negativos) pueden producir chargebacks, reclamos y distorsionar KPIs financieros.",
    "Calidad de datos deficiente (product_id y order_line mismatches) impide atribuir correctamente motivo/canal/SKU y bloquea automatizaciones.",
    "Posible abuso por clientes con devoluciones repetidas; si no se detecta, supone pérdida económica y requiere escalado a Prevención de pérdidas."
  ],
  "accion_7_dias": "Prioridad alta — acciones operativas y de datos esta semana (owner y plazo):\n1) Data / BI (48 h): extraer report semanal Iluminación (últimas 8 semanas vs mismo periodo anterior) con: unidades vendidas, unidades devueltas, importe reembolsado, tasa de devolución (unidades y importe) y top 20 SKUs por devoluciones + desglose por motivo Rxx y canal (owner: Data; plazo: 48 h). \n2) Almacén / Calidad (48–72 h): retener y separar unidades devueltas de top 5 SKUs (P1038, P1086, P1057, P1016, P1013) para inspección física, fotos y test funcional; documentar fallos (owner: Operaciones/Almacén; plazo: 72 h).\n3) Producto / E‑commerce (72 h): revisar y corregir fichas de los top 10 SKUs (dimensiones, fotos, naming, atributo 'voluminoso') según taxonomía; desplegar correcciones CMS si aplica (owner: Producto/E‑commerce; plazo: 72 h).\n4) Logística / Transportistas (72 h): analizar guías/envíos de pedidos afe

Wrote C:\Repos\frasohome-agents-foundry\outputs\orchestrator_response.json

## 10. Revisar salidas generadas

Las ejecuciones guardan resultados en `outputs/` para poder enseñarlos o compararlos.

In [17]:
for path in sorted((REPO_ROOT / "outputs").glob("*")):
    print(path.name, path.stat().st_size, "bytes")

local_data_quality_report.json 13610 bytes
local_data_quality_report.md 1208 bytes


## 11. Cierre de sesión

Puntos para comentar al final:

- File Search aporta grounding documental sobre la KB.
- Code Interpreter calcula métricas y evita inventar cifras.
- El orquestador separa contexto, cálculo, interpretación y síntesis.
- La salida final debe citar fuentes, declarar riesgos y proponer una acción medible.
- Las trazas y tool calls son parte de la demo, no un detalle técnico secundario.